In [ ]:
import random
import os
import numpy as np
import torch

def seed_everything(seed=42):
    # 1. Base Python & OS environment
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

    # 2. PyTorch (CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # For multi-GPU setups

    # 3. Force CUDA deterministic algorithms
    # Note: This ensures identical results but can slightly reduce execution speed
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Call it before loading any models or data
seed_everything(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q resemblyzer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 136.1 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
from tqdm import tqdm

from resemblyzer import VoiceEncoder, preprocess_wav
from sklearn.metrics.pairwise import cosine_similarity

import pandas as pd
import numpy as np

# =====================================
# Load Resemblyzer once
# =====================================

encoder = VoiceEncoder()

# =====================================
# Embedding
# =====================================

def get_resemblyzer_embedding(audio_path):

    wav = preprocess_wav(audio_path)

    embedding = encoder.embed_utterance(
        wav
    )

    return embedding

# =====================================
# Similarity
# =====================================

def resemblyzer_similarity(
    emb1,
    emb2
):

    score = cosine_similarity(
        emb1.reshape(1, -1),
        emb2.reshape(1, -1)
    )[0][0]

    return float(score)

# =====================================
# Folder Evaluation
# =====================================

def evaluate_resemblyzer_similarity(
    reference_audio,
    generated_dir,
    output_csv=None
):

    generated_dir = Path(
        generated_dir
    )

    generated_files = sorted(
        generated_dir.glob("*.wav")
    )

    print(
        f"Found {len(generated_files)} files"
    )

    # -------------------------
    # Reference embedding ONCE
    # -------------------------

    print(
        "Computing reference embedding..."
    )

    ref_embedding = (
        get_resemblyzer_embedding(
            reference_audio
        )
    )

    results = []

    # -------------------------
    # Compare all files
    # -------------------------

    for audio_file in tqdm(
        generated_files,
        desc="Computing similarity"
    ):

        try:

            gen_embedding = (
                get_resemblyzer_embedding(
                    str(audio_file)
                )
            )

            similarity = (
                resemblyzer_similarity(
                    ref_embedding,
                    gen_embedding
                )
            )

            results.append({

                "filename":
                    audio_file.name,

                "similarity":
                    similarity

            })

        except Exception as e:

            print(
                f"Failed: {audio_file}"
            )

            print(e)

    # -------------------------
    # DataFrame
    # -------------------------

    df = pd.DataFrame(
        results
    )

    # -------------------------
    # Save
    # -------------------------

    if output_csv is not None:

        df.to_csv(
            output_csv,
            index=False
        )

        print(
            f"\nSaved CSV:\n{output_csv}"
        )

    # -------------------------
    # Summary
    # -------------------------

    print("\n" + "=" * 80)
    print("RESEMBLYZER SPEAKER SIMILARITY")
    print("=" * 80)

    print(
        f"Files processed: "
        f"{len(df)}"
    )

    print()

    print(
        f"Mean   : "
        f"{df['similarity'].mean():.4f}"
    )

    print(
        f"Median : "
        f"{df['similarity'].median():.4f}"
    )

    print(
        f"Std    : "
        f"{df['similarity'].std():.4f}"
    )

    print(
        f"Min    : "
        f"{df['similarity'].min():.4f}"
    )

    print(
        f"Max    : "
        f"{df['similarity'].max():.4f}"
    )

    print("\n========== WORST 10 ==========\n")

    print(
        df.sort_values(
            "similarity"
        ).head(10)
    )

    return df

Loaded the voice encoder model on cuda in 0.42 seconds.


In [ ]:
df_vc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/test_16k/vc_jennifer",
    output_csv="jen_netraul_vs_jen_vc_resemblyzer"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:22<00:00, 18.03it/s]


Saved CSV:
jen_netraul_vs_jen_vc_resemblyzer

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.7440
Median : 0.7474
Std    : 0.0298
Min    : 0.6489
Max    : 0.8131

========== WORST 10 ==========

                          filename  similarity
80     5322_7679_000025_000005.wav    0.648947
263  7800_283493_000018_000000.wav    0.653967
43    250_142286_000035_000001.wav    0.658857
295       78_368_000021_000001.wav    0.662049
14    250_142276_000006_000005.wav    0.664337
102    5322_7680_000048_000000.wav    0.664792
197   6209_34601_000114_000001.wav    0.665220
221  7800_283478_000036_000001.wav    0.669320
124   6209_34599_000018_000003.wav    0.670699
13    250_142276_000006_000004.wav    0.671417


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/jennifer/jennifer_evc_sad",
    output_csv="jen_neutral_vs_jen_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:22<00:00, 17.80it/s]


Saved CSV:
jen_neutral_vs_jen_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.7218
Median : 0.7282
Std    : 0.0427
Min    : 0.5501
Max    : 0.8262

========== WORST 10 ==========

                          filename  similarity
106    5322_7680_000057_000000.wav    0.550063
166   6209_34601_000065_000001.wav    0.570041
94     5322_7680_000023_000000.wav    0.571236
330       78_369_000064_000000.wav    0.589534
167   6209_34601_000065_000007.wav    0.595972
280  7800_283493_000070_000000.wav    0.608578
177   6209_34601_000084_000001.wav    0.617714
163   6209_34601_000052_000002.wav    0.619473
102    5322_7680_000048_000000.wav    0.619797
26    250_142286_000006_000000.wav    0.622411


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_sad_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/jennifer/jennifer_evc_sad",
    output_csv="jen_sad_vs_jen_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:23<00:00, 16.74it/s]


Saved CSV:
jen_sad_vs_jen_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8223
Median : 0.8288
Std    : 0.0324
Min    : 0.6790
Max    : 0.8869

========== WORST 10 ==========

                          filename  similarity
335       78_369_000068_000000.wav    0.678957
268  7800_283493_000029_000000.wav    0.692759
204   6209_34601_000156_000001.wav    0.694594
202   6209_34601_000153_000002.wav    0.703740
106    5322_7680_000057_000000.wav    0.707481
13    250_142276_000006_000004.wav    0.709967
315       78_369_000036_000001.wav    0.722445
14    250_142276_000006_000005.wav    0.738854
147   6209_34600_000026_000001.wav    0.739355
136   6209_34600_000007_000005.wav    0.739707


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/jennifer/jennifer_evc_angry",
    output_csv="jen_neutral_vs_jen_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:16<00:00, 24.53it/s]


Saved CSV:
jen_neutral_vs_jen_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.6730
Median : 0.6777
Std    : 0.0331
Min    : 0.5148
Max    : 0.7464

========== WORST 10 ==========

                          filename  similarity
365  8312_279790_000039_000001.wav    0.514770
124   6209_34599_000018_000003.wav    0.525562
175   6209_34601_000080_000001.wav    0.556696
177   6209_34601_000084_000001.wav    0.562615
91     5322_7680_000019_000001.wav    0.571397
158   6209_34601_000042_000002.wav    0.581778
152   6209_34601_000008_000001.wav    0.592357
111    5322_7680_000061_000006.wav    0.593369
119   6209_34599_000008_000005.wav    0.593430
235  7800_283492_000010_000001.wav    0.594836


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_angry_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/jennifer/jennifer_evc_angry",
    output_csv="jen_angry_vs_jen_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:20<00:00, 19.79it/s]


Saved CSV:
jen_angry_vs_jen_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8338
Median : 0.8389
Std    : 0.0285
Min    : 0.7215
Max    : 0.8949

========== WORST 10 ==========

                          filename  similarity
176   6209_34601_000084_000000.wav    0.721498
124   6209_34599_000018_000003.wav    0.725022
123   6209_34599_000018_000002.wav    0.728433
111    5322_7680_000061_000006.wav    0.729788
36    250_142286_000028_000000.wav    0.734621
136   6209_34600_000007_000005.wav    0.745789
177   6209_34601_000084_000001.wav    0.747734
175   6209_34601_000080_000001.wav    0.752414
14    250_142276_000006_000005.wav    0.754387
257  7800_283493_000009_000001.wav    0.756304


In [ ]:
df_vc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/test_16k/vc_david",
    output_csv="david_netraul_vs_david_vc_resemblyzer"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:23<00:00, 16.95it/s]


Saved CSV:
david_netraul_vs_david_vc_resemblyzer

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8490
Median : 0.8534
Std    : 0.0282
Min    : 0.7519
Max    : 0.9074

========== WORST 10 ==========

                          filename  similarity
124   6209_34599_000018_000003.wav    0.751949
295       78_368_000021_000001.wav    0.752712
183   6209_34601_000096_000033.wav    0.755553
14    250_142276_000006_000005.wav    0.761441
221  7800_283478_000036_000001.wav    0.763237
331       78_369_000065_000001.wav    0.771568
222  7800_283478_000037_000001.wav    0.772274
121   6209_34599_000017_000003.wav    0.775545
13    250_142276_000006_000004.wav    0.777051
285       78_368_000008_000000.wav    0.779291


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/david/david_evc_sad",
    output_csv="david_neutral_vs_david_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:16<00:00, 24.25it/s]


Saved CSV:
david_neutral_vs_david_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.7762
Median : 0.7818
Std    : 0.0314
Min    : 0.6614
Max    : 0.8436

========== WORST 10 ==========

                          filename  similarity
204   6209_34601_000156_000001.wav    0.661418
36    250_142286_000028_000000.wav    0.666031
91     5322_7680_000019_000001.wav    0.673218
167   6209_34601_000065_000007.wav    0.676252
244  7800_283492_000026_000000.wav    0.690199
161   6209_34601_000045_000001.wav    0.694853
318       78_369_000043_000008.wav    0.699530
278  7800_283493_000067_000001.wav    0.701796
148   6209_34600_000027_000001.wav    0.703585
116   6209_34599_000003_000000.wav    0.705840


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_sad_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/david/david_evc_sad",
    output_csv="david_sad_vs_david_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:23<00:00, 17.19it/s]


Saved CSV:
david_sad_vs_david_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8333
Median : 0.8369
Std    : 0.0269
Min    : 0.7280
Max    : 0.9020

========== WORST 10 ==========

                          filename  similarity
36    250_142286_000028_000000.wav    0.728010
91     5322_7680_000019_000001.wav    0.743365
161   6209_34601_000045_000001.wav    0.751249
221  7800_283478_000036_000001.wav    0.759458
204   6209_34601_000156_000001.wav    0.762640
58     5322_7678_000006_000014.wav    0.763029
167   6209_34601_000065_000007.wav    0.765827
318       78_369_000043_000008.wav    0.769417
244  7800_283492_000026_000000.wav    0.769807
261  7800_283493_000013_000001.wav    0.772091


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/david/david_evc_angry",
    output_csv="david_neutral_vs_david_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:17<00:00, 23.52it/s]


Saved CSV:
david_neutral_vs_david_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.6364
Median : 0.6411
Std    : 0.0278
Min    : 0.5450
Max    : 0.7011

========== WORST 10 ==========

                          filename  similarity
244  7800_283492_000026_000000.wav    0.544950
88     5322_7680_000014_000000.wav    0.550885
204   6209_34601_000156_000001.wav    0.552113
34    250_142286_000024_000001.wav    0.552508
124   6209_34599_000018_000003.wav    0.559170
327       78_369_000056_000002.wav    0.559321
385  8312_279791_000030_000002.wav    0.563324
207  7800_283478_000011_000001.wav    0.568737
272  7800_283493_000052_000000.wav    0.569124
104    5322_7680_000052_000001.wav    0.573091


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_angry_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/david/david_evc_angry",
    output_csv="david_angry_vs_david_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:16<00:00, 23.77it/s]


Saved CSV:
david_angry_vs_david_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8624
Median : 0.8661
Std    : 0.0234
Min    : 0.7850
Max    : 0.9106

========== WORST 10 ==========

                          filename  similarity
207  7800_283478_000011_000001.wav    0.784967
36    250_142286_000028_000000.wav    0.785016
244  7800_283492_000026_000000.wav    0.789373
158   6209_34601_000042_000002.wav    0.789605
78     5322_7679_000022_000000.wav    0.796495
348  8312_279790_000012_000000.wav    0.796843
128   6209_34599_000023_000007.wav    0.797457
88     5322_7680_000014_000000.wav    0.802853
150   6209_34600_000029_000006.wav    0.803711
121   6209_34599_000017_000003.wav    0.805943


In [ ]:
df_vc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/test_16k/vc_morgan",
    output_csv="morgan_netraul_vs_morgan_vc_resemblyzer"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:21<00:00, 18.53it/s]


Saved CSV:
morgan_netraul_vs_morgan_vc_resemblyzer

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8592
Median : 0.8620
Std    : 0.0259
Min    : 0.7743
Max    : 0.9060

========== WORST 10 ==========

                          filename  similarity
150   6209_34600_000029_000006.wav    0.774261
295       78_368_000021_000001.wav    0.785834
199   6209_34601_000134_000000.wav    0.785888
158   6209_34601_000042_000002.wav    0.787903
115    5322_7680_000064_000000.wav    0.788268
125   6209_34599_000020_000001.wav    0.790318
386  8312_279791_000033_000000.wav    0.791620
144   6209_34600_000024_000002.wav    0.791874
374  8312_279791_000010_000002.wav    0.792295
135   6209_34600_000007_000004.wav    0.792858


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/morgan/morgan_evc_sad",
    output_csv="morgan_neutral_vs_morgan_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:16<00:00, 24.42it/s]


Saved CSV:
morgan_neutral_vs_morgan_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.7391
Median : 0.7439
Std    : 0.0338
Min    : 0.5013
Max    : 0.8156

========== WORST 10 ==========

                          filename  similarity
133   6209_34600_000007_000002.wav    0.501314
176   6209_34601_000084_000000.wav    0.589775
268  7800_283493_000029_000000.wav    0.601853
339       78_369_000069_000010.wav    0.605873
272  7800_283493_000052_000000.wav    0.625634
150   6209_34600_000029_000006.wav    0.633056
32    250_142286_000015_000003.wav    0.641051
193   6209_34601_000104_000001.wav    0.641691
62     5322_7678_000006_000026.wav    0.651954
58     5322_7678_000006_000014.wav    0.663320


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_sad_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/morgan/morgan_evc_sad",
    output_csv="morgan_sad_vs_morgan_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:22<00:00, 17.54it/s]


Saved CSV:
morgan_sad_vs_morgan_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8242
Median : 0.8313
Std    : 0.0379
Min    : 0.4224
Max    : 0.8908

========== WORST 10 ==========

                          filename  similarity
133   6209_34600_000007_000002.wav    0.422364
176   6209_34601_000084_000000.wav    0.653345
62     5322_7678_000006_000026.wav    0.673632
150   6209_34600_000029_000006.wav    0.682700
115    5322_7680_000064_000000.wav    0.694951
193   6209_34601_000104_000001.wav    0.698724
212  7800_283478_000020_000002.wav    0.724293
123   6209_34599_000018_000002.wav    0.730348
106    5322_7680_000057_000000.wav    0.736780
167   6209_34601_000065_000007.wav    0.741497


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/morgan/morgan_evc_angry",
    output_csv="morgan_neutral_vs_morgan_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [01:42<00:00,  3.89it/s]


Saved CSV:
morgan_neutral_vs_morgan_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.7178
Median : 0.7223
Std    : 0.0258
Min    : 0.5978
Max    : 0.7802

========== WORST 10 ==========

                          filename  similarity
268  7800_283493_000029_000000.wav    0.597826
123   6209_34599_000018_000002.wav    0.606638
186   6209_34601_000096_000057.wav    0.629789
239  7800_283492_000016_000001.wav    0.630874
145   6209_34600_000024_000003.wav    0.646990
166   6209_34601_000065_000001.wav    0.651808
210  7800_283478_000018_000002.wav    0.651957
348  8312_279790_000012_000000.wav    0.652239
307       78_369_000020_000001.wav    0.652588
138   6209_34600_000009_000002.wav    0.653363


In [ ]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_angry_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/evc_test_16k/morgan/morgan_evc_angry",
    output_csv="morgan_angry_vs_morgan_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:16<00:00, 23.57it/s]


Saved CSV:
morgan_angry_vs_morgan_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8743
Median : 0.8787
Std    : 0.0239
Min    : 0.7428
Max    : 0.9231

========== WORST 10 ==========

                          filename  similarity
123   6209_34599_000018_000002.wav    0.742840
166   6209_34601_000065_000001.wav    0.776479
268  7800_283493_000029_000000.wav    0.789183
239  7800_283492_000016_000001.wav    0.798007
124   6209_34599_000018_000003.wav    0.806715
116   6209_34599_000003_000000.wav    0.808407
144   6209_34600_000024_000002.wav    0.809739
170   6209_34601_000068_000009.wav    0.809911
136   6209_34600_000007_000005.wav    0.809957
210  7800_283478_000018_000002.wav    0.812300
